In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset import CounterfactualAudioDataset
from models import AudioTextCounterfactualModel

/home/deodato/.pyenv/versions/xai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class CounterfactualLoss(nn.Module):
    """
    Composite Loss function combining Factual Consistency and Angle Loss
    as defined in Equation (6) of the paper.
    """
    def __init__(self, margin=0.1, w1=1.0, w2=100.0):
        super().__init__()
        self.margin = margin
        self.w1 = w1  # Angle Loss Weight
        self.w2 = w2  # Factual Consistency Weight
        self.mse = nn.MSELoss()

    def forward(self, audio_embeds, factual_embeds, counterfactual_embeds):
        # Equation 5: Factual Consistency Loss (Mean Squared Error)
        # Drives the audio embedding towards the factual caption
        l_factual = self.mse(audio_embeds, factual_embeds)
        
        # Equation 3 & 4: Angle Loss (Triplet Margin Cosine)
        # Cosine similarity is the dot product of L2 normalized vectors
        cos_factual = torch.sum(audio_embeds * factual_embeds, dim=-1)
        cos_counterfactual = torch.sum(audio_embeds * counterfactual_embeds, dim=-1)
        
        # Punishes the model if the counterfactual similarity is greater than the factual similarity
        # L_angle = max(0, cos(a, cf) - cos(a, f) + margin)
        l_angle = torch.mean(torch.clamp(cos_counterfactual - cos_factual + self.margin, min=0.0))
        
        # Equation 6: Total Composite Loss
        total_loss = (self.w1 * l_angle) + (self.w2 * l_factual)
        
        return total_loss, l_angle, l_factual

In [3]:
def evaluate_retrieval(model, dataloader, device):
    """
    Evaluates the model on the Language-Based Audio Retrieval task (Table 2).
    """
    model.eval()
    all_audio_embeds = []
    all_text_embeds = []
    
    print("Extracting embeddings for Evaluation...")
    with torch.no_grad():
        for batch in tqdm(dataloader):
            log_mel_specs, captions, _ = batch
            log_mel_specs = log_mel_specs.to(device)
            
            # Extract
            audio_embeds = model.encode_audio(log_mel_specs)
            text_embeds = model.encode_text(captions, device)
            
            all_audio_embeds.append(audio_embeds.cpu())
            all_text_embeds.append(text_embeds.cpu())
            
    # Concatenate all embeddings for the entire evaluation set
    all_audio_embeds = torch.cat(all_audio_embeds, dim=0) # Shape: (N, 512)
    all_text_embeds = torch.cat(all_text_embeds, dim=0)   # Shape: (N, 512)
    
    # Compute Cosine Similarity Matrix (N_text_queries, N_audio_database)
    # Because vectors are normalized, matrix multiplication yields cosine similarity
    sim_matrix = torch.matmul(all_text_embeds, all_audio_embeds.T)
    
    N = sim_matrix.shape[0]
    top1_correct = 0
    top10_correct = 0
    
    # Calculate Top-k Recall
    for i in range(N):
        # Sort audio database by similarity to the current text query
        ranked_indices = torch.argsort(sim_matrix[i], descending=True)
        
        # The correct audio for text query `i` is at index `i`
        if ranked_indices[0] == i:
            top1_correct += 1
            
        if i in ranked_indices[:10]:
            top10_correct += 1
            
    top1_acc = top1_correct / N
    top10_acc = top10_correct / N
    
    print(f"Evaluation Results -> Top-1: {top1_acc:.4f} | Top-10: {top10_acc:.4f}")
    return top1_acc, top10_acc

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters from paper (Table 4 optimal setup)
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
W1 = 1.0     # Angle Loss weight
W2 = 100.0   # Factual Consistency weight

# 1. Load Datasets
# Replace these paths with your generated CSVs from the data collection steps
print("Initializing Datasets...")
train_dataset = CounterfactualAudioDataset("data/metadata.csv") # Combined train CSV
# test_dataset = CounterfactualAudioDataset("data/clotho_eval_metadata.csv") 

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 2. Load Model & Optimizer
model = AudioTextCounterfactualModel().to(device)
criterion = CounterfactualLoss(margin=0.1, w1=W1, w2=W2)

# We only train the audio encoder (adapter + resnet)
optimizer = torch.optim.AdamW(model.audio_encoder.parameters(), lr=LR)

# 3. Training Loop
print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss_epoch = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in progress_bar:
        log_mel_specs, factual_captions, counterfactual_captions = batch
        log_mel_specs = log_mel_specs.to(device)
        
        # Forward pass: Extract audio features
        audio_embeds = model.encode_audio(log_mel_specs)
        
        # Forward pass: Extract text features using frozen CLIP
        factual_embeds = model.encode_text(factual_captions, device)
        counterfactual_embeds = model.encode_text(counterfactual_captions, device)
        
        # Calculate composite loss
        loss, l_angle, l_factual = criterion(audio_embeds, factual_embeds, counterfactual_embeds)
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss_epoch += loss.item()
        progress_bar.set_postfix({"Loss": loss.item()})
        
    print(f"Epoch {epoch+1} Completed | Average Loss: {total_loss_epoch/len(train_loader):.4f}")
    
    # Optional: Run evaluation after every epoch
    # evaluate_retrieval(model, test_loader, device)
    
# Save the trained Audio Encoder weights
torch.save(model.audio_encoder.state_dict(), "counterfactual_audio_encoder.pth")
print("Training Finished. Model weights saved.")

Using device: cuda
Initializing Datasets...


Loading weights: 100%|██████████| 196/196 [00:00<00:00, 55474.97it/s]
CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weig

Starting Training...


Epoch 1/10: 100%|██████████| 1303/1303 [04:51<00:00,  4.46it/s, Loss=0.143]


Epoch 1 Completed | Average Loss: 0.1510


Epoch 2/10:   0%|          | 0/1303 [00:00<?, ?it/s]